# Proyecto LSP — Exportar el modelo a TensorFlow.js (para la demo web)

Convierte `modelo_pucp305_final.keras` → **TensorFlow.js GraphModel** para que la demo
HTML (`web/lsp_demo.html`) corra el modelo en el navegador, sin Python.

Genera un `web_model.zip` que descargas y descomprimes en `web/web_model/` (junto al HTML).

> **Por qué reconstruye el modelo con `unroll=True`:** el LSTM normal usa un bucle
> `while` (control flow dinámico) que TF.js no logra ejecutar como GraphModel
> (errores *"dynamic op Exit"* / *"Cannot compute the outputs [Identity]"*). Con
> `unroll=True` los 30 pasos se expanden en un grafo estático, sin bucle, y TF.js sí
> lo ejecuta. Los pesos son idénticos (se transfieren con `set_weights`), y una
> verificación confirma que las salidas coinciden con el modelo original.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# tensorflowjs instala sus propias deps, INCLUIDA jax (su converter la importa).
# OJO: a diferencia de los notebooks de inferencia (04/05/06-setup), aquí NO se
# desinstala jax — hacerlo rompe `import tensorflowjs` (ModuleNotFoundError: jax).
!pip install -q tensorflowjs

In [ ]:
import shutil, subprocess
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dropout, Dense

M  = Path('/content/drive/MyDrive/PUCP305_models')
SM = Path('/content/saved_model')
OUT = Path('/content/web_model')
if OUT.exists(): shutil.rmtree(OUT)   # limpio: no arrastrar shards viejos
OUT.mkdir(parents=True)

old = tf.keras.models.load_model(M / 'modelo_pucp305_final.keras', compile=False)
n_clases = old.output_shape[-1]
print('Modelo original:', old.input_shape, '->', old.output_shape)

# Reconstruir IDENTICO pero con unroll=True: expande los 30 pasos del LSTM en un
# grafo estatico (sin tf.while_loop -> sin ops Enter/Exit). Asi TF.js puede
# ejecutarlo; el bucle while era lo que rompia la inferencia en el navegador.
# unroll NO cambia los pesos, asi que set_weights() transfiere 1:1.
new = Sequential([
    Input((30, 258)),
    Bidirectional(LSTM(128, return_sequences=True, unroll=True)),
    Dropout(0.4),
    Bidirectional(LSTM(64, unroll=True)),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dense(n_clases, activation='softmax'),
])
new.set_weights(old.get_weights())

# Sanity check: el modelo desenrollado debe dar las MISMAS salidas que el original
x = np.random.rand(2, 30, 258).astype('float32')
diff = float(np.abs(old.predict(x, verbose=0) - new.predict(x, verbose=0)).max())
print(f'Max diff original vs unroll: {diff:.2e}  (debe ser ~0)')
assert diff < 1e-4, 'Los pesos no se transfirieron bien'

# Exportar -> SavedModel -> GraphModel (ya sin control flow dinamico)
if SM.exists(): shutil.rmtree(SM)
new.export(str(SM))
cmd = ['tensorflowjs_converter', '--input_format=tf_saved_model',
       '--output_format=tfjs_graph_model',
       '--signature_name=serving_default', '--saved_model_tags=serve',
       str(SM), str(OUT)]
print('\n$', ' '.join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout); print(r.stderr)
r.check_returncode()
print('GraphModel (unroll) OK.')

shutil.copy(M / 'classes_pucp305.json', OUT / 'classes_pucp305.json')
print('\nArchivos generados:')
for p in sorted(OUT.iterdir()):
    print(f'  {p.name:30} {p.stat().st_size/1024:8.1f} KB')

In [ ]:
# Empaquetar y descargar
shutil.make_archive('/content/web_model', 'zip', OUT)
from google.colab import files
files.download('/content/web_model.zip')
print('Descomprime web_model.zip dentro de la carpeta web/ del repo (junto a lsp_demo.html).')